# RAG: Baseline Retrieval vs. Cross-Encoder Re-Ranking

Two retrieval pipelines run over the **same chunked corpus** and are compared:

- **Pipeline 1 — Baseline:** dense bi-encoder retrieval. Embed the query, pull the top 10 chunks by cosine similarity, hand the top 3 to the LLM.
- **Pipeline 2 — Re-ranking:** take the same top 10, re-score every `(query, chunk)` pair with a cross-encoder, re-sort, and hand the new top 3 to the LLM.

The corpus is the Catan base-game rulebook (rules + almanac) — dense with exceptions, penalties and role limits, which makes retrieval mistakes easy to see.

## 1. Setup & Environment

Models used:
- Bi-encoder: `sentence-transformers/all-MiniLM-L6-v2`
- Cross-encoder: `cross-encoder/ms-marco-MiniLM-L-6-v2`
- Answer generation: `Qwen/Qwen2.5-1.5B-Instruct`, run locally with Hugging Face `transformers`

All three models download from Hugging Face the first time they are used and then run on the local CPU. No API key or network service is required.

In [ ]:
import os, re, time, textwrap, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import pypdf
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

# Retrieval settings
BI_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
TOP_K = 10   # candidates pulled by the baseline
TOP_N = 3    # chunks actually sent to the LLM

# Chunking settings
CHUNK_CHARS = 700
CHUNK_OVERLAP = 120

# Answer generation (local Hugging Face model)
LOCAL_MODEL = os.environ.get("LOCAL_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
print("Answer model (local):", LOCAL_MODEL)

## 2. Dataset

A single complex PDF is the knowledge base. The notebook downloads it on first run and keeps it in `data/`.

Each page is extracted separately so every chunk can be traced back to a page number.

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
PDF_PATH = DATA_DIR / "catan_rules.pdf"
PDF_URL = "https://world-of-board-games.com.sg/docs/Catan-5ed.pdf"

if not PDF_PATH.exists():
    print("Downloading corpus ...")
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
print("PDF size (KB):", PDF_PATH.stat().st_size // 1024)


def load_pages(path):
    reader = pypdf.PdfReader(str(path))
    if reader.is_encrypted:
        reader.decrypt("")
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text).strip()
        if text:
            pages.append({"page": i + 1, "text": text})
    return pages


pages = load_pages(PDF_PATH)
print(f"Loaded {len(pages)} non-empty pages")
print(pages[1]["text"][:400])

## 3. Chunking

The pages are split into overlapping character windows. Paragraph boundaries are respected where possible so a rule and its exception tend to stay together, and a small overlap keeps context from being cut mid-sentence. Each chunk records its source page.

In [ ]:
def chunk_pages(pages, size=CHUNK_CHARS, overlap=CHUNK_OVERLAP):
    chunks = []
    for item in pages:
        page_no, text = item["page"], item["text"]
        paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
        buf = ""
        for para in paragraphs:
            if len(buf) + len(para) + 1 <= size:
                buf = (buf + " " + para).strip()
            else:
                if buf:
                    chunks.append({"page": page_no, "text": buf})
                if len(para) > size:
                    # a single long paragraph: slide a window across it
                    for start in range(0, len(para), size - overlap):
                        chunks.append({"page": page_no, "text": para[start:start + size]})
                    buf = ""
                else:
                    tail = buf[-overlap:] if buf else ""
                    buf = (tail + " " + para).strip()
        if buf:
            chunks.append({"page": page_no, "text": buf})
    return chunks


chunks = chunk_pages(pages)
chunk_texts = [c["text"] for c in chunks]
print(f"Total chunks: {len(chunks)}")
print(f"Average chunk length: {sum(len(t) for t in chunk_texts) // len(chunk_texts)} chars")
print("\nExample chunk:\n", chunk_texts[5])

## 4. Pipeline 1 — Baseline Retrieval (Bi-Encoder)

The bi-encoder embeds each chunk independently into a vector. Vectors are L2-normalised and stored in a FAISS inner-product index, so an inner-product search is equivalent to cosine similarity. For a query we embed it the same way and pull the `TOP_K` nearest chunks.

In [ ]:
bi_encoder = SentenceTransformer(BI_ENCODER_NAME)

chunk_embeddings = bi_encoder.encode(
    chunk_texts, normalize_embeddings=True, show_progress_bar=True
).astype("float32")

dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)
print(f"Indexed {index.ntotal} chunks, dimension {dim}")


def baseline_retrieve(query, k=TOP_K):
    q = bi_encoder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    return [{"id": int(i), "score": float(s)} for i, s in zip(ids[0], scores[0])]

## 5. Pipeline 2 — Re-Ranking (Cross-Encoder)

The cross-encoder takes the `TOP_K` baseline candidates and reads each `(query, chunk)` pair **together** in one pass, so its score reflects direct interaction between the query and the passage rather than two separate embeddings. We re-sort the 10 candidates by this score and keep the new top 3.

In [ ]:
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME)


def rerank(query, candidates):
    ids = [c["id"] for c in candidates]
    pairs = [(query, chunk_texts[i]) for i in ids]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(
        ({"id": i, "score": float(s)} for i, s in zip(ids, scores)),
        key=lambda x: x["score"], reverse=True,
    )
    return ranked

## 6. Prompt & Answer Generation

Both pipelines use the **identical prompt structure** — only the three retrieved chunks differ. The model is told to answer strictly from the passages and to say so when the answer is absent, which is what exposes a weak retrieval: if the critical chunk is missing, the answer is wrong or incomplete.

The model is loaded once (lazily) and reused. Greedy decoding (`do_sample=False`) keeps the output deterministic, so the two pipelines are compared on their retrieved context, not on sampling noise.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

_tokenizer = None
_model = None


def load_llm():
    global _tokenizer, _model
    if _model is None:
        _tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)
        _model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL, dtype=torch.float32)
        _model.eval()
    return _tokenizer, _model


SYSTEM_PROMPT = (
    "You answer board-game rules questions using only the provided context passages. "
    "Quote the specific rule and cite the passage numbers you used. "
    "If the answer is not contained in the passages, say that you cannot find it."
)


def format_context(ids):
    blocks = []
    for rank, i in enumerate(ids, start=1):
        blocks.append(f"[{rank}] (page {chunks[i]['page']}) {chunk_texts[i]}")
    return "\n\n".join(blocks)


def build_user_prompt(query, ids):
    return f"Context passages:\n{format_context(ids)}\n\nQuestion: {query}\n\nAnswer:"


def generate_answer(query, ids, max_new_tokens=256):
    tokenizer, model = load_llm()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(query, ids)},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    )
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    prompt_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(output[0][prompt_len:], skip_special_tokens=True).strip()

## 7. End-to-end check on a straightforward query

A quick sanity run of both pipelines on an easy question, where the baseline is already expected to do well.

In [ ]:
def run_both(query):
    base = baseline_retrieve(query, TOP_K)
    base_ids = [c["id"] for c in base]
    reranked = rerank(query, base)
    rr_ids = [c["id"] for c in reranked]
    return base, base_ids, reranked, rr_ids


def show_candidates(title, items):
    print(title)
    for rank, c in enumerate(items, start=1):
        preview = chunk_texts[c["id"]][:90].replace("\n", " ")
        print(f"  {rank:>2}. chunk {c['id']:>3} (p{chunks[c['id']]['page']}) "
              f"score={c['score']:.3f}  {preview}")


demo_query = "What does a player need to do to win the game?"
base, base_ids, reranked, rr_ids = run_both(demo_query)
show_candidates("Baseline top 10:", base)
print()
show_candidates("Re-ranked top 10:", reranked)

print("\n--- Answer 1 (baseline top 3) ---")
print(generate_answer(demo_query, base_ids[:TOP_N]))
print("\n--- Answer 2 (re-ranked top 3) ---")
print(generate_answer(demo_query, rr_ids[:TOP_N]))

## 8. Failure Mode Analysis (Stress Testing)

Each query below is chosen so the critical chunk sits **outside** the baseline top 3 (buried at ranks 4–10, or missed on wording) but the cross-encoder promotes it into the top 3. Categories covered: keyword trap / distractor, exception & corner case, and negation / constraint.

In [ ]:
FAILURE_QUERIES = [
    {
        "category": "Keyword trap / distractor",
        "query": "What happens when a seven is rolled?",
        "note": "The baseline is pulled toward passages that only share wording - the number-token "
                "text that says 'there is no 7', and the table of contents - and misses the actual "
                "robber-and-discard procedure.",
    },
    {
        "category": "Exception & corner case",
        "query": "Can I build a settlement right next to another settlement?",
        "note": "The generic 'how to build a settlement' text ranks high; the Distance Rule "
                "restriction that actually answers the question sits further down.",
    },
    {
        "category": "Negation / constraint",
        "query": "Can I give away resource cards to another player as a gift?",
        "note": "The governing rule is a prohibition ('you cannot give away cards'); the bi-encoder "
                "blurs the negative and surfaces generic trading text instead.",
    },
    {
        "category": "Buried specific (ranks 4-10)",
        "query": "How many points do I need to win?",
        "note": "The exact victory-point total appears in one worked example, buried below several "
                "passages that mention points in general.",
    },
]


def analyse(query, note="", category="", make_answers=True):
    base = baseline_retrieve(query, TOP_K)
    base_ids = [c["id"] for c in base]
    reranked = rerank(query, base)
    rr_ids = [c["id"] for c in reranked]

    print("=" * 100)
    if category:
        print(f"[{category}]")
    print("Q:", query)
    if note:
        print("   ", note)
    print("\nBaseline order (rank: chunk, page, score):")
    for rank, c in enumerate(base, start=1):
        mark = "  <= dropped by baseline top-3" if rank > TOP_N else ""
        print(f"  {rank:>2}: chunk {c['id']:>3} p{chunks[c['id']]['page']} {c['score']:.3f}{mark}")

    print("\nRe-ranked order:")
    for rank, c in enumerate(reranked, start=1):
        print(f"  {rank:>2}: chunk {c['id']:>3} p{chunks[c['id']]['page']} {c['score']:.3f}")

    promoted = [i for i in rr_ids[:TOP_N] if i in base_ids[TOP_N:]]
    records = []
    for pid in promoted:
        before = base_ids.index(pid) + 1
        after = rr_ids.index(pid) + 1
        records.append({"query": query, "category": category, "chunk": pid,
                        "page": chunks[pid]["page"],
                        "baseline_rank": before, "rerank_rank": after})
        print(f"\nPROMOTED chunk {pid}: baseline #{before} -> re-rank #{after} (page {chunks[pid]['page']})")
        print("   ", chunk_texts[pid][:240].replace("\n", " "))

    answers = {}
    if make_answers:
        answers["baseline"] = generate_answer(query, base_ids[:TOP_N])
        answers["rerank"] = generate_answer(query, rr_ids[:TOP_N])
        print("\n--- Answer 1 (baseline top 3) ---\n", answers["baseline"])
        print("\n--- Answer 2 (re-ranked top 3) ---\n", answers["rerank"])
    return records


rank_shift_records = []
for item in FAILURE_QUERIES:
    rank_shift_records += analyse(item["query"], item["note"], item["category"])
    print()

## 9. Rank-Shift Summary

The chunks that the cross-encoder lifted from ranks 4–10 into the top 3. These are exactly the passages the baseline would have withheld from the LLM.

In [ ]:
rank_shift_df = pd.DataFrame(rank_shift_records,
                             columns=["query", "category", "chunk", "page",
                                      "baseline_rank", "rerank_rank"])
rank_shift_df

## 10. Latency Profiling

Wall-clock time of the baseline retrieval versus the full re-ranking pipeline (baseline retrieval **plus** the cross-encoder pass). Retrieval-only timing is measured — the LLM call is excluded because it is identical for both pipelines. Each query is run several times and averaged.

In [ ]:
def time_baseline(query, repeats=5):
    t0 = time.perf_counter()
    for _ in range(repeats):
        baseline_retrieve(query, TOP_K)
    return (time.perf_counter() - t0) / repeats


def time_rerank_pipeline(query, repeats=5):
    t0 = time.perf_counter()
    for _ in range(repeats):
        cand = baseline_retrieve(query, TOP_K)
        rerank(query, cand)
    return (time.perf_counter() - t0) / repeats


latency_rows = []
for item in FAILURE_QUERIES:
    q = item["query"]
    b = time_baseline(q)
    r = time_rerank_pipeline(q)
    latency_rows.append({"query": q[:45] + "...",
                         "baseline_ms": round(b * 1000, 1),
                         "rerank_pipeline_ms": round(r * 1000, 1),
                         "overhead_ms": round((r - b) * 1000, 1),
                         "slowdown_x": round(r / b, 1)})

latency_df = pd.DataFrame(latency_rows)
latency_df

## 11. Discussion pointers

The written analysis — architecture comparison, qualitative answer comparison, latency reading, and the scaling question — is in `report/REPORT.md`, backed by the tables produced above.

In short: the cross-encoder is accurate but costs a forward pass per candidate, so it is used only to re-rank a short baseline shortlist, never to score the whole corpus. Running it over every chunk for every query would multiply cost by the corpus size with no index to prune the search.